# 04b Preprocessing experiments
Additional check, not required by the project: it tests the decisions of `02_preprocessing.ipynb`, the input format and some training options by training, with the chosen settings of `03_model1_embeddings.ipynb` and `04_model2_rnn.ipynb`.

**In short.** Question: does another preprocessing decision, input format or training option clearly lower the validation loss? **[after the run]**

Each variant changes one decision of the baseline; the rest stays the same: the chosen configuration of each model, seed 42, the validation set. The test set is not loaded. `ALL_VARIANTS = False` (section "Model 1") runs only the baseline and what `05_evaluation.ipynb` needs: `hypotheses-only` for both models and `per-choice` for model 1.

| Decision | Baseline | Variants |
|---|---|---|
| Word embeddings | GloVe | `fasttext`, `word2vec`: other vectors, the same preprocessing |
| Tokenization | `word_tokenize` | `whitespace-tokens` |
| Lowercasing | lowercase | `cased` |
| Stemming, lemmatization | none | `stemming`, `lemmatization` (with part-of-speech tags) |
| Stopwords | kept | `no-stopwords` |
| Punctuation | kept | `no-punctuation` |
| Unknown words | skipped | `unk-token`: one fixed vector, the average of the training words |
| Rare words | kept | `no-rare-words`: fewer than 5 occurrences in the training data |
| Vocabulary | any GloVe word | `train-vocab` |
| Format cleaning | none | `format-cleaning`: curly and broken apostrophes become `'` |
| Truncation | model 1: none; model 2: 34 tokens | `truncate-20`, `truncate-10` |
| Feature selection | O1, O2, h1, h2 | `hypotheses-only`, `observations-only`; model 1 also `hypothesis-difference`: O1, O2, h1 - h2 |
| Input format | O1, O2, h1, h2 in fixed positions, 2 outputs | `per-choice`: the same classifier scores each hypothesis; model 2 also `one-pass`: one RNN pass per hypothesis over O1, O2, separator, hypothesis |
| Weight decay | none, Adam | `weight-decay-0.001`, `weight-decay-0.01`, `weight-decay-0.1`: AdamW |
| Loss | cross-entropy | `margin`; `contrastive`, which also changes the classifier |

**Decision rule:** a variant is looked at only if it has a lower validation loss and significantly more correct predictions than the baseline: the variant right on more examples than the baseline, p < 0.05 in the exact McNemar test (as in `04_model2_rnn.ipynb`). The requirement of more correct predictions was added after the first model 1 runs on the CPU, because the two-sided test alone could accept a significantly worse variant; it can only make the rule stricter. One run per variant and about 23 comparisons per model, so small differences and one p below 0.05 can be chance. `margin` and `contrastive` have their own loss on another scale: it selects their best epoch, but only their correct predictions are compared with the baseline. The output lists them apart, and W&B logs their loss under its own names (`own_train_loss`, `own_val_loss`, `best_own_val_loss`). `hypotheses-only` and `observations-only` are diagnostics, not candidates: the task is to choose the hypothesis that explains the observations. The validation set is unfiltered (see "Data" in `README.md`), so it can also reward decisions that exploit artifacts of the hypotheses.

**GPU version (1.11):** with the settings chosen on the GPU: model 1 with batch size 2048, model 2 with layer norm and batch size 128, patience 10, deterministic mode. The vocabulary of `02_preprocessing.ipynb` also holds the validation words, so the variant `glove-vocab` is replaced by `train-vocab`. New: the word embeddings, weight decay, two other losses, and a stopped run can continue (section "Training options"). **[after the run]** marks the results to fill in.

In [ ]:
# setup: on Colab, connect Google Drive; everywhere, use the GPU if there is one
import os
import sys

# on a GPU, a seed alone does not repeat a run exactly; this variable must be set before the GPU is used
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:2"

import torch

torch.backends.cudnn.deterministic = True
# deterministic operations where they exist, and a warning for the others
torch.use_deterministic_algorithms(True, warn_only=True)

if "google.colab" in sys.modules:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/nlp-project1-art")
    %pip install -q gensim datasets wandb nltk
    # the key comes from Colab Secrets, so it is not in the notebook
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
# the W&B server of this project
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Data, GloVe and resources
The training and validation splits (like on slide 9 of the course slides "Course Project HS26"), the GloVe vectors and the NLTK resources, as in `02_preprocessing.ipynb`. New is the part-of-speech tagger for the lemmatization.

In [ ]:
import re
import string
import time

import gensim.downloader as api
import nltk
import numpy as np
from datasets import load_dataset

train = load_dataset("amarfurt/art-nlp-hs26", split="train")
valid = load_dataset("amarfurt/art-nlp-hs26", split="validation")
print(len(train), len(valid))

wv = api.load("glove-wiki-gigaword-300")

nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)


## Preprocessing variants
`preprocess()` is the function of `02_preprocessing.ipynb` with one change per variant. The tagger is loaded once, because `nltk.pos_tag` loads it again in every call. `is_known()` decides which words have a vector in a variant; the others are skipped, or replaced by the `<unk>` vector in `unk-token`. `embeddings()` gives the word vectors of a variant: `fasttext` and `word2vec`, the other two embeddings the project allows, change only the vectors, so the text stays lowercased, which suits GloVe: word2vec covers 77.62% of the lowercased training tokens, fastText 99.11%. Both are downloaded once (about 2.7 GB) and take several GB of memory.

In [ ]:
columns = ["observation_1", "observation_2", "hypothesis_1", "hypothesis_2"]

stopwords = set(nltk.corpus.stopwords.words("english"))
stemmer = nltk.stem.PorterStemmer()
lemmatizer = nltk.stem.WordNetLemmatizer()
tagger = nltk.tag.perceptron.PerceptronTagger()
# first letter of the part-of-speech tag -> part of speech of WordNet (default: noun)
wordnet_pos = {"J": "a", "V": "v", "R": "r"}

def is_punctuation(word):
    return all(character in string.punctuation for character in word)

# tokens of each text are kept, because most variants need the same ones
token_cache = {}

def tokenize(text):
    if text not in token_cache:
        token_cache[text] = nltk.word_tokenize(text)
    return token_cache[text]

# the baseline, with one difference per variant
def preprocess(text, variant):
    if variant == "format-cleaning":
        # curly and broken apostrophes become '
        text = re.sub("[\u2019\u00b4\ufffd]+", "'", text)
    words = text.split() if variant == "whitespace-tokens" else tokenize(text)
    if variant == "cased":
        return words
    words = [word.lower() for word in words]
    if variant == "stemming":
        return [stemmer.stem(word) for word in words]
    if variant == "lemmatization":
        return [lemmatizer.lemmatize(word, wordnet_pos.get(tag[0], "n")) for word, tag in tagger.tag(words)]
    if variant == "no-stopwords":
        return [word for word in words if word not in stopwords]
    if variant == "no-punctuation":
        return [word for word in words if not is_punctuation(word)]
    return words

# preprocessed words of all texts of a split, per column
def split_words(split, variant):
    return {column: [preprocess(text, variant) for text in split[column]] for column in columns}

# how often each word occurs in the training data of this variant
def word_counts(train_words):
    return nltk.FreqDist(word for column in columns for words in train_words[column] for word in words)

# the other word embeddings of the project, with the names of 02_preprocessing.ipynb; loaded when they are first used
other_embeddings = {"fasttext": "fasttext-wiki-news-subwords-300", "word2vec": "word2vec-google-news-300"}
loaded_embeddings = {}

# the word vectors of a variant: GloVe, except in the variants fasttext and word2vec
def embeddings(variant):
    if variant not in other_embeddings:
        return wv
    if variant not in loaded_embeddings:
        loaded_embeddings[variant] = api.load(other_embeddings[variant])
    return loaded_embeddings[variant]

# does this word have a vector in this variant?
def is_known(word, variant, counts):
    if word not in embeddings(variant):
        return False
    if variant == "no-rare-words":
        return counts[word] >= 5
    # vocabulary of the training data: variant of both models
    if variant == "train-vocab":
        return word in counts
    return True

# the texts that a variant uses
def used_columns(variant):
    return {"hypotheses-only": columns[2:], "observations-only": columns[:2]}.get(variant, columns)

print(preprocess("He didn\u2019t go to the hospitals.", "baseline"))
print(preprocess("He didn\u2019t go to the hospitals.", "format-cleaning"))
print(preprocess("He didn't go to the hospitals.", "lemmatization"))


## Classes, evaluation and W&B
The classes, `evaluate()` and the loss as in `03_model1_embeddings.ipynb` and `04_model2_rnn.ipynb`. `RNNClassifier` has two changes that do not affect the baseline: `num_texts`, because the feature selection variants have 2 texts, and `.clamp(min=1)`, because a variant can leave a text without any word. `evaluate()` also takes the loss as an argument and returns the predictions, as in `05_evaluation.ipynb`, so that the accuracy and the McNemar test use the same predictions.

In [ ]:
import os

# no W&B log lines in the notebook output; the metrics are still logged
os.environ["WANDB_SILENT"] = "true"

import torch
import torch.nn as nn
import wandb
from scipy.stats import binomtest
from torch.nn.utils.rnn import pack_padded_sequence
from torch.optim import Adam, AdamW
from torch.utils.data import DataLoader, TensorDataset

# same class as in 03_model1_embeddings.ipynb
class Classifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

# as in 04_model2_rnn.ipynb, with num_texts and clamp(min=1)
class RNNClassifier(nn.Module):

    def __init__(self, embedding_matrix, rnn_type="lstm", hidden_dim=128, num_layers=2, dropout=0.0, inner_dim=100, num_texts=4):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=True, padding_idx=0)
        embedding_dim = embedding_matrix.shape[1]
        if rnn_type == "lstm":
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        else:
            self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = Classifier(num_texts * hidden_dim, 2, inner_dim=inner_dim)
        self.num_texts = num_texts

    def forward(self, x):
        features = []
        for i in range(self.num_texts):
            text = x[:, i]
            # real length of each text, at least 1
            lengths = (text != 0).sum(dim=1).clamp(min=1)
            embedded = self.embedding(text)
            packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
            _, hidden = self.rnn(packed)
            if isinstance(self.rnn, nn.LSTM):
                hidden = hidden[0]
            features.append(hidden[-1])
        features = torch.concat(features, dim=-1)
        features = self.dropout(features)
        return self.classifier(features)

criterion = nn.CrossEntropyLoss()

# same function as in 03_model1_embeddings.ipynb, with the loss as an argument; it also returns the predictions, as in 05_evaluation.ipynb
def evaluate(model, loader, loss_function=criterion):
    model.eval()
    losses, predictions, labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            output = model(x)
            losses.append(loss_function(output, y).item())
            predictions.append(output.argmax(dim=1))
            labels.append(y)
    predictions = torch.cat(predictions)
    labels = torch.cat(labels)
    accuracy = (predictions == labels).float().mean().item()
    return np.mean(losses), accuracy, predictions

# labels: 0 = h1, 1 = h2
y_train = torch.tensor([example - 1 for example in train["label"]]).to(device)
y_valid = torch.tensor([example - 1 for example in valid["label"]]).to(device)

wandb.login()

# one line per variant, with the exact McNemar test against the baseline: first the variants with cross-entropy, sorted by the validation loss,
# then margin and contrastive: their loss is on another scale, so it is not ranked with the others
def print_results(results):
    correct_baseline = results["baseline"]["predictions"] == y_valid
    ranked = sorted((item for item in results.items() if not item[1]["own_loss"]), key=lambda item: item[1]["loss"])
    own_loss = [item for item in results.items() if item[1]["own_loss"]]
    for variant, result in ranked + own_loss:
        correct = result["predictions"] == y_valid
        # examples that only one of the two gets right
        only_variant = (correct & ~correct_baseline).sum().item()
        only_baseline = (~correct & correct_baseline).sum().item()
        p = binomtest(only_variant, only_variant + only_baseline, 0.5).pvalue if only_variant + only_baseline > 0 else 1.0
        loss_label = "own loss" if result["own_loss"] else "valid loss"
        print(f"{variant}: {loss_label} {result['loss']:.4f}, valid acc {result['accuracy']:.2%}, best epoch {result['epoch']}, only variant right {only_variant}, only baseline right {only_baseline}, p = {p:.4f}")

## Input format variants
The baseline gives h1 and h2 fixed input positions, so its prediction can depend on the position (`04_model2_rnn.ipynb`, section "Check: does the position of the hypotheses matter?"). Question: **does scoring each hypothesis separately lower the validation loss?** The output stays 2 logits, so the loss, the training and the evaluation do not change, and swapping h1 and h2 swaps the two logits, as the check below shows.
- `per-choice` (both models): the same 2-layer classifier scores [O1, O2, h1] and [O1, O2, h2], with 1 output each (the pairs on slide 25 and the one score per choice on slide 26 of the course slides "Project Discussions" (Week 5); for averaged GloVe vectors also the bag-of-words baseline of Bhagavatula et al. (2020, Appendix A.4), 50.5%). In model 2, the RNN still reads each text separately.
- `one-pass` (model 2): the RNN reads [O1, O2, separator, h1] and [O1, O2, separator, h2], one pass each (the RNN example on slide 26). The same classifier scores each final hidden state.

In [ ]:
# one score per hypothesis: the same classifier scores each hypothesis
class PerChoiceClassifier(nn.Module):

    def __init__(self, text_dim, inner_dim, num_observations=2):
        super().__init__()
        # size of the vector of one text (model 1: 300, model 2: RNN hidden size)
        self.text_dim = text_dim
        # how many observation vectors come before h1 and h2 (0 = none)
        self.num_observations = num_observations
        # input: the observations and one hypothesis; output: 1 score
        self.scorer = Classifier((num_observations + 1) * text_dim, 1, inner_dim=inner_dim)

    def forward(self, x):
        # x side by side: the observations, h1, h2
        start = self.num_observations * self.text_dim
        observations = x[:, :start]
        h1 = x[:, start:start + self.text_dim]
        h2 = x[:, start + self.text_dim:]
        # the same scorer for both hypotheses: [batch size, 1] each
        score1 = self.scorer(torch.concat([observations, h1], dim=1))
        score2 = self.scorer(torch.concat([observations, h2], dim=1))
        # the two scores are the 2 logits: [batch size, 2], as in the baseline
        return torch.concat([score1, score2], dim=1)


In [ ]:
# untrained classifier on 5 made-up examples: 4 texts of 300 numbers side by side
torch.manual_seed(42)
test_model = PerChoiceClassifier(300, inner_dim=100)
x = torch.randn(5, 1200)
# the same examples with h1 and h2 swapped
swapped = torch.concat([x[:, :600], x[:, 900:], x[:, 600:900]], dim=1)
with torch.no_grad():
    logits = test_model(x)
    swapped_logits = test_model(swapped)
# 2 logits per example, as in the baseline
print(logits.shape)
# swapping the hypotheses must swap the logits
print(torch.equal(logits.flip(1), swapped_logits))
# number of parameters: per-choice (900 inputs, 1 output, used twice) and baseline (1200 inputs, 2 outputs)
print(sum(p.numel() for p in test_model.parameters()), sum(p.numel() for p in Classifier(1200, 2, inner_dim=100).parameters()))


## Training options
Options that `03_model1_embeddings.ipynb` and `04_model2_rnn.ipynb` do not test, from the course slides "Project Discussions" (Week 6), as variants of the baseline.
- `weight-decay-0.001`, `weight-decay-0.01`, `weight-decay-0.1`: AdamW instead of Adam, because the two only differ with weight decay (slide 31). 0.01 is the default of `torch.optim.AdamW`, in exponential steps (same slides (Week 7), slides 52 and 53). Weight decay should be "not larger than gradient * learning rate" (slide 34): the learning rate is on both sides, so the check in each model section compares the decay and the gradient on one batch, a rough check, because Adam scales its steps.
- `margin` (slide 30): `nn.MultiMarginLoss` on the same 2 logits.
- `contrastive` (slide 30): two 2-layer ReLU networks turn the observations and each hypothesis into a vector of 100 numbers; the score of a hypothesis is its cosine similarity with the observations, with the formula of `nn.CosineEmbeddingLoss`. The classifier changes too, so this does not show the effect of the loss alone.
- **Continuing a stopped run** (slides 35 and 37): `fit()`, the training loop of all variants, saves the model, the optimizer, the best values, the curve and the random number states to a `-state.pth` file after every epoch. Running the sweep again continues a stopped run after its last finished epoch, in a new W&B run with the curve so far; the stopped run stays as crashed, and finished runs are trained again. If the settings have changed, the run starts again. The file is deleted when the run ends.

The check in each model section trains the same 4 epochs three times: twice without a stop, and once with a stop after epoch 2, continued by a new model with other start weights and random numbers. All three must end with the same weights (deterministic mode: PyTorch documentation "Reproducibility").

In [ ]:
# the variant margin: the logit of the correct hypothesis must be above the other one by at least 1
margin_loss = nn.MultiMarginLoss()

# the variant contrastive: the score of a hypothesis is its cosine similarity with the observations
class ContrastiveClassifier(nn.Module):

    def __init__(self, text_dim, inner_dim, output_dim=100):
        super().__init__()
        # size of the vector of one text (model 1: 300, model 2: RNN hidden size)
        self.text_dim = text_dim
        # 2-layer networks: O1 and O2 together, and one hypothesis, each to a vector of output_dim numbers
        self.observations = Classifier(2 * text_dim, output_dim, inner_dim=inner_dim)
        self.hypothesis = Classifier(text_dim, output_dim, inner_dim=inner_dim)

    def forward(self, x):
        # x side by side: O1, O2, h1, h2
        observations = self.observations(x[:, :2 * self.text_dim])
        h1 = self.hypothesis(x[:, 2 * self.text_dim:3 * self.text_dim])
        h2 = self.hypothesis(x[:, 3 * self.text_dim:])
        # one score per hypothesis: [batch size, 2]; the higher score is the prediction, as with the 2 logits of the baseline
        return torch.stack([torch.cosine_similarity(observations, h1), torch.cosine_similarity(observations, h2)], dim=1)

# the formula of nn.CosineEmbeddingLoss on these scores: correct hypothesis 1 - score, wrong hypothesis the score if it is above 0
def contrastive_loss(scores, y):
    examples = torch.arange(len(y), device=y.device)
    correct = scores[examples, y]
    wrong = scores[examples, 1 - y]
    # divided by 2, because each example gives 2 pairs
    return ((1 - correct) + wrong.clamp(min=0)).mean() / 2

# layer norm in the classifier of the chosen model 2, as in 04_model2_rnn.ipynb: after the first linear layer of every 2-layer network
def add_layer_norm(model, inner_dim):
    for part in list(model.classifier.modules()):
        if isinstance(part, Classifier):
            part.linear1 = nn.Sequential(part.linear1, nn.LayerNorm(inner_dim))

# the optimizer and the loss of a variant
def training_options(model, variant, learning_rate):
    # weight decay: AdamW; all other variants: Adam, as in the baseline
    if variant.startswith("weight-decay-"):
        optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=float(variant.split("-")[-1]))
    else:
        optimizer = Adam(model.parameters(), lr=learning_rate)
    # margin and contrastive: their own loss; all other variants: cross-entropy, as in the baseline
    loss_function = {"margin": margin_loss, "contrastive": contrastive_loss}.get(variant, criterion)
    return optimizer, loss_function

In [ ]:
# untrained classifier on 5 made-up examples: 4 texts of 300 numbers side by side
torch.manual_seed(42)
test_model = ContrastiveClassifier(300, inner_dim=100)
x = torch.randn(5, 1200)
y = torch.tensor([0, 1, 1, 0, 1])
with torch.no_grad():
    scores = test_model(x)
    # the same examples with h1 and h2 swapped: swapping the hypotheses must swap the scores
    swapped = torch.concat([x[:, :600], x[:, 900:], x[:, 600:900]], dim=1)
    print(torch.equal(scores.flip(1), test_model(swapped)))
    # contrastive_loss must be the same as nn.CosineEmbeddingLoss on the 2 pairs of each example
    observations = test_model.observations(x[:, :600])
    h1 = test_model.hypothesis(x[:, 600:900])
    h2 = test_model.hypothesis(x[:, 900:])
    # target 1: the correct hypothesis, target -1: the wrong one
    targets = torch.concat([1 - 2 * y, 2 * y - 1])
    pairs = nn.CosineEmbeddingLoss()(torch.concat([observations, observations]), torch.concat([h1, h2]), targets)
    print(torch.isclose(contrastive_loss(scores, y), pairs))
# margin_loss on the logits 1.0 and 0.5, the first one correct: (1 - 1.0 + 0.5) / 2 classes = 0.25
print(margin_loss(torch.tensor([[1.0, 0.5]]), torch.tensor([0])))

In [ ]:
# the training loop of all variants, as in the sweeps of 03_model1_embeddings.ipynb and 04_model2_rnn.ipynb,
# with the state of the run saved after every epoch, so that a stopped run can continue
def fit(model, optimizer, loss_function, train_loader, train_eval_loader, valid_loader, checkpoint_path, settings, max_epochs, patience, stop_after=None):
    state_path = checkpoint_path.replace(".pth", "-state.pth")
    best_valid_loss = float("inf")
    best_valid_acc, best_epoch = 0.0, 0
    epochs_without_improvement = 0
    # one row per finished epoch
    history = []
    # margin and contrastive: their loss is on another scale, so it has its own names in W&B and is not charted or ranked with the cross-entropy
    train_key, valid_key = ("train_loss", "val_loss") if loss_function is criterion else ("own_train_loss", "own_val_loss")

    # a state file is only left by a run that did not finish
    if os.path.exists(state_path):
        # on the CPU, because the random number states must be there; load_state_dict moves the rest to the model
        state = torch.load(state_path, map_location="cpu", weights_only=True)
        if state["settings"] == settings:
            model.load_state_dict(state["model"])
            optimizer.load_state_dict(state["optimizer"])
            best_valid_loss, best_valid_acc, best_epoch = state["best_valid_loss"], state["best_valid_acc"], state["best_epoch"]
            epochs_without_improvement = state["epochs_without_improvement"]
            history = state["history"]
            torch.set_rng_state(state["rng"])
            if "cuda_rng" in state and torch.cuda.is_available():
                torch.cuda.set_rng_state_all(state["cuda_rng"])
            # the W&B run is new, so it gets the curve so far
            for row in history:
                wandb.log(row)
            print(f"{checkpoint_path}: continues after epoch {len(history)}")

    first_epoch = len(history)
    # the last finished epoch, if the loop below has nothing left to do
    epoch = first_epoch - 1
    for epoch in range(first_epoch, max_epochs):
        model.train()
        for x, y in train_loader:
            output = model(x)
            loss = loss_function(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc, _ = evaluate(model, train_eval_loader, loss_function)
        valid_loss, valid_acc, _ = evaluate(model, valid_loader, loss_function)
        # float(): a NumPy number cannot be loaded with weights_only=True
        row = {"epoch": epoch + 1, train_key: float(train_loss), "train_acc": train_acc, valid_key: float(valid_loss), "val_acc": valid_acc}
        wandb.log(row)
        history.append(row)
        if valid_loss < best_valid_loss:
            best_valid_loss = float(valid_loss)
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                break
        state = {"settings": settings, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                 "best_valid_loss": best_valid_loss, "best_valid_acc": best_valid_acc, "best_epoch": best_epoch,
                 "epochs_without_improvement": epochs_without_improvement, "history": history, "rng": torch.get_rng_state()}
        if torch.cuda.is_available():
            state["cuda_rng"] = torch.cuda.get_rng_state_all()
        # first to another file: a session that ends during the saving leaves the last complete state
        torch.save(state, state_path + ".tmp")
        os.replace(state_path + ".tmp", state_path)
        # only for the check below: stop here, as if the session had ended
        if epoch + 1 == stop_after:
            return None

    # the run has ended, so its state is not needed any more
    if os.path.exists(state_path):
        os.remove(state_path)
    return best_valid_loss, best_valid_acc, best_epoch, epoch


# the size of the weight decay, and a stopped run compared with runs that did not stop
def check_training_options(build_model, X_train, X_valid, batch_size, learning_rate):
    X_train, X_valid = X_train.to(device), X_valid.to(device)

    # one batch of an untrained model
    torch.manual_seed(42)
    model = build_model()
    criterion(model(X_train[:batch_size]), y_train[:batch_size]).backward()
    # only the trained parameters: the frozen GloVe vectors have no gradient
    parameters = [p for p in model.parameters() if p.grad is not None]
    weights = torch.cat([p.flatten() for p in parameters]).norm().item()
    gradient = torch.cat([p.grad.flatten() for p in parameters]).norm().item()
    for weight_decay in [0.001, 0.01, 0.1]:
        print(f"weight decay {weight_decay}: decay {weight_decay * weights:.4f}, gradient {gradient:.4f}")
    # AdamW must not change the frozen GloVe vectors
    if hasattr(model, "embedding"):
        before = model.embedding.weight.clone()
        AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1).step()
        print("GloVe vectors unchanged by AdamW:", torch.equal(before, model.embedding.weight))

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True, drop_last=True)
    train_eval_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=317)
    valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=100)

    def one_run(path, seed=42, stop_after=None):
        torch.manual_seed(seed)
        model = build_model()
        optimizer = Adam(model.parameters(), lr=learning_rate)
        fit(model, optimizer, criterion, train_loader, train_eval_loader, valid_loader, path, "check", 4, 4, stop_after)
        return model

    def same_weights(model1, model2):
        return all(torch.equal(a, b) for a, b in zip(model1.state_dict().values(), model2.state_dict().values()))

    # no W&B run for this check; teardown: start without the run id of an earlier sweep run
    wandb.teardown()
    wandb.init(mode="disabled")
    first = one_run("checkpoints/check-first.pth")
    repeat = one_run("checkpoints/check-repeat.pth")
    # stopped after epoch 2; a new model with other start weights and other random numbers continues
    one_run("checkpoints/check-stopped.pth", stop_after=2)
    continued = one_run("checkpoints/check-stopped.pth", seed=0)
    wandb.finish()
    print("same when repeated:", same_weights(first, repeat))
    print("same after a stop:", same_weights(first, continued))
    for path in ["checkpoints/check-first.pth", "checkpoints/check-repeat.pth", "checkpoints/check-stopped.pth"]:
        os.remove(path)

## Model 1
The chosen configuration of `03_model1_embeddings.ipynb`: 600 hidden units, learning rate 0.000316, batch size 2048, patience 10. A W&B sweep (method `grid`) trains all variants. The baseline must reproduce the features saved in `02_preprocessing.ipynb` and the chosen run (best epoch 40, validation loss 0.6460, accuracy 64.00%).

In [ ]:
# one vector per text: the average of its word vectors, as in 02_preprocessing.ipynb
def model1_features(split_word_lists, variant, counts, unk_vector):
    max_length = {"truncate-20": 20, "truncate-10": 10}.get(variant)
    blocks = []
    for column in used_columns(variant):
        block = []
        for words in split_word_lists[column]:
            vectors = []
            for word in words:
                if is_known(word, variant, counts):
                    vectors.append(embeddings(variant)[word])
                elif variant == "unk-token":
                    vectors.append(unk_vector)
            # [:None] keeps all words
            vectors = vectors[:max_length]
            block.append(np.mean(vectors, axis=0) if vectors else np.zeros(wv.vector_size))
        blocks.append(np.array(block, dtype=np.float32))
    if variant == "hypothesis-difference":
        # O1, O2, h1 - h2
        blocks = [blocks[0], blocks[1], blocks[2] - blocks[3]]
    return torch.tensor(np.concatenate(blocks, axis=1))

# all features of a variant
def prepare_model1(variant):
    train_words = split_words(train, variant)
    valid_words = split_words(valid, variant)
    counts = word_counts(train_words)
    # fixed vector for unknown words: the average of the vectors of the training words
    unk_vector = None
    if variant == "unk-token":
        unk_vector = np.mean([wv[word] for word in counts if word in wv], axis=0)
    return model1_features(train_words, variant, counts, unk_vector), model1_features(valid_words, variant, counts, unk_vector)

# check: the baseline must be the same as the saved features of model 1
X_train, X_valid = prepare_model1("baseline")
saved = np.load("features/model1.npz")
print("Baseline equals the saved features:", np.array_equal(X_train.numpy(), saved["X_train"]), np.array_equal(X_valid.numpy(), saved["X_valid"]))


In [ ]:
# False: only the baseline and the runs that 05_evaluation.ipynb needs; True: all variants
ALL_VARIANTS = False

variants_model1 = ["baseline", "hypotheses-only", "per-choice"]
if ALL_VARIANTS:
    variants_model1 += ["whitespace-tokens", "cased", "stemming", "lemmatization", "no-stopwords", "no-punctuation",
                        "unk-token", "no-rare-words", "train-vocab", "format-cleaning", "truncate-20", "truncate-10",
                        "observations-only", "hypothesis-difference",
                        "weight-decay-0.001", "weight-decay-0.01", "weight-decay-0.1", "margin", "contrastive", "fasttext", "word2vec"]

sweep_config_model1 = {
    "name": "model1-preprocessing",
    "method": "grid",
    "metric": {"name": "best_val_loss", "goal": "minimize"},
    "parameters": {"variant": {"values": variants_model1}},
}

# results of all variants, for the comparison below
results_model1 = {}

# the batch size of the chosen run in 03_model1_embeddings.ipynb
BATCH_SIZE1 = 2048

# one run: the W&B agent calls this function once per variant
def run_model1(inner_dim=600, learning_rate=0.000316, batch_size=BATCH_SIZE1, max_epochs=300, patience=10):
    run = wandb.init(project="nlp-project1-art-gpu")
    variant = run.config["variant"]
    name = f"prep-mlp-{variant}"
    run.name = name
    run.config.update({"model": "prep-mlp", "inner_dim": inner_dim, "learning_rate": learning_rate, "batch_size": batch_size, "max_epochs": max_epochs, "patience": patience, "seed": 42})
    checkpoint_path = f"checkpoints/{name}.pth"

    X_train, X_valid = prepare_model1(variant)
    X_train, X_valid = X_train.to(device), X_valid.to(device)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True, drop_last=True)
    train_eval_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=317)
    valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=100)

    # same training as in the sweep of 03_model1_embeddings.ipynb
    torch.manual_seed(42)
    if variant == "per-choice":
        # the same features, only the classifier differs
        model = PerChoiceClassifier(wv.vector_size, inner_dim=inner_dim).to(device)
    elif variant == "contrastive":
        # the same features, the classifier compares the observations with each hypothesis
        model = ContrastiveClassifier(wv.vector_size, inner_dim=inner_dim).to(device)
    else:
        model = Classifier(X_train.shape[1], 2, inner_dim=inner_dim).to(device)
    optimizer, loss_function = training_options(model, variant, learning_rate)
    settings = f"{inner_dim} {learning_rate} {batch_size} {max_epochs} {patience}"
    best_valid_loss, best_valid_acc, best_epoch, epoch = fit(model, optimizer, loss_function, train_loader, train_eval_loader, valid_loader, checkpoint_path, settings, max_epochs, patience)

    # margin and contrastive: their loss is on another scale, so W&B must not rank it with the cross-entropy
    wandb.run.summary["best_val_loss" if loss_function is criterion else "best_own_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    # predictions of the best epoch, from the same batches as its accuracy, for the comparison with the baseline
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    _, _, predictions = evaluate(model, valid_loader, loss_function)
    results_model1[variant] = {"loss": best_valid_loss, "accuracy": best_valid_acc, "epoch": best_epoch, "predictions": predictions, "own_loss": loss_function is not criterion}
    loss_label = "valid loss" if loss_function is criterion else "own loss"
    print(f"{name}: best epoch {best_epoch} of {epoch + 1}, {loss_label} {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")

In [ ]:
# the check of the section "Training options", with the settings of the baseline
check_training_options(lambda: Classifier(1200, 2, inner_dim=600).to(device), X_train, X_valid, BATCH_SIZE1, 0.000316)

In [ ]:
# register the sweep at W&B and train all variants (a grid ends by itself)
sweep_id = wandb.sweep(sweep_config_model1, project="nlp-project1-art-gpu")
wandb.agent(sweep_id, function=run_model1)


In [ ]:
print_results(results_model1)


### Result for model 1
**[after the run]**

## Model 2
The chosen configuration of `04_model2_rnn.ipynb`: GRU with layer norm in the classifier, RNN hidden size 128, learning rate 0.001, dropout 0.5, batch size 128, patience 10, at most 100 epochs. All variants have layer norm. The vocabulary, the embedding matrix and the padding are built per variant as in `02_preprocessing.ipynb`. The baseline must reproduce the saved input of model 2 and the chosen run (best epoch 6, validation loss 0.5867, accuracy 66.70%).

In [ ]:
# token indices of all texts of a split: shape [examples, number of texts, max_length]
def model2_indices(split_word_lists, variant, vocab, unk_index, max_length):
    rows = []
    for column in used_columns(variant):
        block = []
        for words in split_word_lists[column]:
            numbers = []
            for word in words:
                if word in vocab:
                    numbers.append(vocab[word])
                elif variant == "unk-token":
                    numbers.append(unk_index)
            numbers = numbers[:max_length]
            block.append(numbers + [0] * (max_length - len(numbers)))
        rows.append(block)
    return torch.tensor(np.array(rows, dtype=np.int64).transpose(1, 0, 2))

# input and embedding matrix of a variant
def prepare_model2(variant):
    train_words = split_words(train, variant)
    valid_words = split_words(valid, variant)
    counts = word_counts(train_words)
    # all candidate words, the most frequent training words first, as in 02_preprocessing.ipynb
    candidates = [word for word, count in counts.most_common()]
    # then the validation words in the order they appear: the embeddings are pretrained, so this uses no labels
    candidates += [word for column in columns for words in valid_words[column] for word in words]
    vocab = {}
    for word in candidates:
        if word not in vocab and is_known(word, variant, counts):
            vocab[word] = len(vocab) + 1
    # one more index for <unk> in the variant unk-token, and for the separator in the variant one-pass
    unk_index = len(vocab) + 1
    size = len(vocab) + 2 if variant in ["unk-token", "one-pass"] else len(vocab) + 1
    matrix = np.zeros((size, wv.vector_size), dtype=np.float32)
    for word, number in vocab.items():
        matrix[number] = embeddings(variant)[word]
    # the rows of the training words: the validation words must not change the training input
    train_rows = [number for word, number in vocab.items() if word in counts]
    if variant == "unk-token":
        # fixed vector for unknown words: the average of the training words, as for model 1
        matrix[unk_index] = matrix[train_rows].mean(axis=0)
    if variant == "one-pass":
        # the frozen separator: one fixed random vector with the spread of the vectors of the training words
        matrix[unk_index] = np.random.default_rng(42).normal(0, matrix[train_rows].std(), wv.vector_size)
        # one sequence per hypothesis: O1, O2, separator, hypothesis; 62 = longest training sequence
        X_train = one_pass_indices(train_words, vocab, unk_index, 62)
        X_valid = one_pass_indices(valid_words, vocab, unk_index, 62)
        return X_train, X_valid, torch.tensor(matrix)
    max_length = {"truncate-20": 20, "truncate-10": 10}.get(variant, 34)
    X_train = model2_indices(train_words, variant, vocab, unk_index, max_length)
    X_valid = model2_indices(valid_words, variant, vocab, unk_index, max_length)
    return X_train, X_valid, torch.tensor(matrix)

# check: the baseline must be the same as the saved input of model 2 (the saved matrix also has the rows of the test words, at the end)
X_train, X_valid, embedding_matrix = prepare_model2("baseline")
saved = np.load("features/model2.npz")
print("Baseline equals the saved input:", np.array_equal(X_train.numpy(), saved["X_train"]), np.array_equal(X_valid.numpy(), saved["X_valid"]), np.array_equal(embedding_matrix.numpy(), saved["embedding_matrix"][:len(embedding_matrix)]))


### Input of the variant `one-pass`
One sequence per hypothesis: the token indices of O1 and O2, the separator and the hypothesis, as in the format on slide 26 of the course slides "Project Discussions" (Week 5). The embeddings are frozen, so the separator is one fixed vector: random, with the same spread as the vectors of the training words. The padding length is 62 tokens, the longest such training sequence, and the padding is at the end, as for the other variants.

In [ ]:
# token indices with one sequence per hypothesis: shape [examples, 2, max_length]
def one_pass_indices(split_word_lists, vocab, separator, max_length):
    rows = []
    # the 4 texts of one example at a time
    for o1, o2, h1, h2 in zip(*[split_word_lists[column] for column in columns]):
        # O1 and O2: only words in the vocabulary, as in model2_indices
        observations = [vocab[word] for word in o1 + o2 if word in vocab]
        sequences = []
        for hypothesis in [h1, h2]:
            # O1, O2, separator, hypothesis
            numbers = observations + [separator] + [vocab[word] for word in hypothesis if word in vocab]
            # cut and pad at the end, as in model2_indices
            numbers = numbers[:max_length]
            sequences.append(numbers + [0] * (max_length - len(numbers)))
        rows.append(sequences)
    return torch.tensor(np.array(rows, dtype=np.int64))


In [ ]:
X_train_one, X_valid_one, matrix_one = prepare_model2("one-pass")
# shape [examples, 2, 62]; the matrix has one row more than the baseline: the separator
print(X_train_one.shape, X_valid_one.shape, matrix_one.shape)
# longest real sequence per split: no validation sequence may be longer than 62, or it would be cut
print((X_train_one != 0).sum(dim=-1).max().item(), (X_valid_one != 0).sum(dim=-1).max().item())
# the separator is the last row of the matrix; every sequence must contain it exactly once
separator = matrix_one.shape[0] - 1
print((X_train_one == separator).sum().item() == 2 * len(train), (X_valid_one == separator).sum().item() == 2 * len(valid))


In [ ]:
variants_model2 = ["baseline", "hypotheses-only"]
if ALL_VARIANTS:
    variants_model2 += ["whitespace-tokens", "cased", "stemming", "lemmatization", "no-stopwords", "no-punctuation",
                        "unk-token", "no-rare-words", "train-vocab", "format-cleaning", "truncate-20", "truncate-10",
                        "observations-only", "per-choice", "one-pass",
                        "weight-decay-0.001", "weight-decay-0.01", "weight-decay-0.1", "margin", "contrastive", "fasttext", "word2vec"]

sweep_config_model2 = {
    "name": "model2-preprocessing",
    "method": "grid",
    "metric": {"name": "best_val_loss", "goal": "minimize"},
    "parameters": {"variant": {"values": variants_model2}},
}

# results of all variants, for the comparison below
results_model2 = {}

# the batch size from the test in 04_model2_rnn.ipynb
BATCH_SIZE2 = 128

# one run: the W&B agent calls this function once per variant
def run_model2(hidden_dim=128, learning_rate=0.001, dropout=0.5, max_epochs=100, patience=10):
    start = time.time()
    run = wandb.init(project="nlp-project1-art-gpu")
    variant = run.config["variant"]
    name = f"prep-gru-{variant}"
    run.name = name
    run.config.update({"model": "prep-gru", "rnn_type": "gru", "hidden_dim": hidden_dim, "num_layers": 2, "inner_dim": 100, "learning_rate": learning_rate, "dropout": dropout, "max_epochs": max_epochs, "patience": patience, "batch_size": BATCH_SIZE2, "seed": 42})
    checkpoint_path = f"checkpoints/{name}.pth"

    X_train, X_valid, embedding_matrix = prepare_model2(variant)
    X_train, X_valid = X_train.to(device), X_valid.to(device)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE2, shuffle=True, drop_last=True)
    train_eval_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=317)
    valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=100)

    # same training as in the sweep of 04_model2_rnn.ipynb
    torch.manual_seed(42)
    model = RNNClassifier(embedding_matrix, rnn_type="gru", hidden_dim=hidden_dim, dropout=dropout, num_texts=X_train.shape[1]).to(device)
    if variant == "per-choice":
        # the same RNN, only the classifier on the final hidden states differs
        model.classifier = PerChoiceClassifier(hidden_dim, inner_dim=100).to(device)
    if variant == "one-pass":
        # 2 sequences, one per hypothesis: the observations are already in each final hidden state
        model.classifier = PerChoiceClassifier(hidden_dim, inner_dim=100, num_observations=0).to(device)
    if variant == "contrastive":
        # the same RNN, the classifier compares the observations with each hypothesis
        model.classifier = ContrastiveClassifier(hidden_dim, inner_dim=100).to(device)
    # the chosen configuration has layer norm in the classifier
    add_layer_norm(model, 100)
    model = model.to(device)
    optimizer, loss_function = training_options(model, variant, learning_rate)
    settings = f"{hidden_dim} {learning_rate} {dropout} {BATCH_SIZE2} {max_epochs} {patience}"
    best_valid_loss, best_valid_acc, best_epoch, epoch = fit(model, optimizer, loss_function, train_loader, train_eval_loader, valid_loader, checkpoint_path, settings, max_epochs, patience)

    # margin and contrastive: their loss is on another scale, so W&B must not rank it with the cross-entropy
    wandb.run.summary["best_val_loss" if loss_function is criterion else "best_own_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    # predictions of the best epoch, from the same batches as its accuracy, for the comparison with the baseline
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    _, _, predictions = evaluate(model, valid_loader, loss_function)
    results_model2[variant] = {"loss": best_valid_loss, "accuracy": best_valid_acc, "epoch": best_epoch, "predictions": predictions, "own_loss": loss_function is not criterion}
    loss_label = "valid loss" if loss_function is criterion else "own loss"
    minutes = (time.time() - start) / 60
    print(f"{name}: best epoch {best_epoch} of {epoch + 1}, {loss_label} {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%} ({minutes:.1f} min)")

In [ ]:
# the check of the section "Training options", with the settings of the baseline
def chosen_model2():
    model = RNNClassifier(embedding_matrix, rnn_type="gru", hidden_dim=128, dropout=0.5)
    add_layer_norm(model, 100)
    return model.to(device)

check_training_options(chosen_model2, X_train, X_valid, BATCH_SIZE2, 0.001)

In [ ]:
# register the sweep at W&B and train all variants (a grid ends by itself)
sweep_id = wandb.sweep(sweep_config_model2, project="nlp-project1-art-gpu")
wandb.agent(sweep_id, function=run_model2)


In [ ]:
print_results(results_model2)


### Result for model 2
**[after the run]**

## All results
**[after the run]**

## Conclusion
**[after the run]**

## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Preprocessing decisions: course slides "Project Discussions" (Week 2), slide 10 (including "map to UNK"), and the course document "Grading criteria HS26" (A. Marfurt); baseline, `preprocess()`, the vocabulary, the padding and the embedding matrix from `02_preprocessing.ipynb` (see the sources there).
- Dataset: course version of ART, `amarfurt/art-nlp-hs26` on Hugging Face (A. Marfurt), based on Bhagavatula et al. (2020), https://arxiv.org/abs/1908.05739; loading the splits from slide 9 of the course slides "Course Project HS26".
- Word embeddings: GloVe (Pennington et al., 2014), loaded with gensim's downloader (Řehůřek & Sojka, 2010) as in the course exercise "Word Embeddings" (Week 2); fastText (Bojanowski et al., 2017) and word2vec (Mikolov et al., 2013), the other embeddings allowed in the Project 1 description, with the same downloader, as in `02_preprocessing.ipynb`.
- Stemming and lemmatization: `nltk.stem.PorterStemmer` and `nltk.stem.WordNetLemmatizer` from the course slides "NLP Pipeline" (Week 1), slides 15 and 16; `lemmatize(word, pos)` from the NLTK documentation, https://www.nltk.org/api/nltk.stem.wordnet.html; part-of-speech tagger `PerceptronTagger` from the NLTK documentation, https://www.nltk.org/api/nltk.tag.perceptron.html, which `nltk.pos_tag` uses, https://www.nltk.org/api/nltk.tag.html; first letters of the Penn Treebank tags (J, V, R), Penn Treebank tag set as listed by `nltk.help.upenn_tagset`, https://www.nltk.org/api/nltk.help.html; downloading the resources with `nltk.download`, NLTK documentation "Installing NLTK Data", https://www.nltk.org/data.html.
- Format cleaning: `re.sub`, Python documentation, https://docs.python.org/3/library/re.html; the affected characters from the format check in `02_preprocessing.ipynb`.
- Models, training and evaluation: `Classifier`, `RNNClassifier` and `evaluate()` from `03_model1_embeddings.ipynb` and `04_model2_rnn.ipynb` (see the sources there); `Tensor.clamp`, PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.clamp.html; `numpy.transpose`, https://numpy.org/doc/stable/reference/generated/numpy.transpose.html, `numpy.array_equal`, https://numpy.org/doc/stable/reference/generated/numpy.array_equal.html, and `dict.get`, https://docs.python.org/3/library/stdtypes.html#dict.get; the rows of the training words with a list of positions, NumPy documentation "Indexing on ndarrays", https://numpy.org/doc/stable/user/basics.indexing.html.
- Sweep: W&B sweeps from the course slides "Project Discussions" (Week 7), slide 57, and the W&B documentation on the sweep configuration, https://docs.coreweave.com/models/sweeps/define-sweep-configuration, its options, https://docs.coreweave.com/models/sweeps/sweep-config-keys, and the sweeps walkthrough, https://docs.coreweave.com/models/sweeps/walkthrough; grid search from the same slides, slide 44.
- Comparison with the baseline: exact McNemar test (McNemar, 1947) with `scipy.stats.binomtest` as in `04_model2_rnn.ipynb` (see the sources there); `sorted(..., key=...)` with a `lambda`, Python documentation "Sorting Techniques", https://docs.python.org/3/howto/sorting.html.
- Correction for multiple comparisons: Holm, S. (1979), *A simple sequentially rejective multiple test procedure*, Scandinavian Journal of Statistics 6(2), 65–70, as described in "Holm–Bonferroni method", Wikipedia, https://en.wikipedia.org/wiki/Holm%E2%80%93Bonferroni_method.
- Input format variants: pairs per choice (slide 25), and one RNN pass per choice with a separator and one score per choice (slide 26), from the previous semester's examples in the course slides "Project Discussions" (Week 5); for averaged GloVe vectors also the bag-of-words baseline of Bhagavatula et al. (2020), Appendix A.4, https://arxiv.org/abs/1908.05739; separators as special tokens from the same slides, (Week 4), slide 15, and (Week 5), slide 20; `torch.concat` as in the course exercise "Sequence-to-sequence RNN" (Week 4); column indexing as in the course exercise "PyTorch" (Week 1); `torch.flip`, https://docs.pytorch.org/docs/stable/generated/torch.flip.html, `torch.randn`, https://docs.pytorch.org/docs/stable/generated/torch.randn.html, and `torch.max`, https://docs.pytorch.org/docs/stable/generated/torch.max.html, from the PyTorch documentation; `numpy.random.default_rng` and `Generator.normal`, NumPy documentation, https://numpy.org/doc/stable/reference/random/generator.html; `ndarray.std`, https://numpy.org/doc/stable/reference/generated/numpy.ndarray.std.html; `zip()` with `*`, Python documentation, https://docs.python.org/3/library/functions.html#zip.
- Training options: margin and contrastive losses (Week 6, slide 30), Adam and AdamW (Week 6, slide 31), weight decay "not larger than gradient * learning rate" (Week 6, slide 34), checkpointing and resuming training (Week 6, slides 35 and 37), exponentially changing values (Week 7, slide 52) and the boundary of the tested range (Week 7, slide 53) from the course slides "Project Discussions". `torch.optim.AdamW` (`weight_decay`, default 0.01), https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html; `nn.MultiMarginLoss`, https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiMarginLoss.html; `nn.CosineEmbeddingLoss` (the formula of `contrastive_loss`), https://docs.pytorch.org/docs/stable/generated/torch.nn.CosineEmbeddingLoss.html; `torch.cosine_similarity`, https://docs.pytorch.org/docs/stable/generated/torch.cosine_similarity.html; `torch.stack`, https://docs.pytorch.org/docs/stable/generated/torch.stack.html; `torch.arange`, https://docs.pytorch.org/docs/stable/generated/torch.arange.html; `torch.cat`, https://docs.pytorch.org/docs/stable/generated/torch.cat.html; `torch.flatten`, https://docs.pytorch.org/docs/stable/generated/torch.flatten.html; `Tensor.norm`, https://docs.pytorch.org/docs/stable/generated/torch.Tensor.norm.html; `torch.isclose`, https://docs.pytorch.org/docs/stable/generated/torch.isclose.html; `str.startswith`, `str.split`, `str.replace` and `dict.get`, Python documentation, https://docs.python.org/3/library/stdtypes.html; `hasattr` and `all`, https://docs.python.org/3/library/functions.html.
- Layer norm in all variants of model 2: as in `04_model2_rnn.ipynb` (see the sources there); `Module.modules`, PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html; `isinstance`, Python documentation, https://docs.python.org/3/library/functions.html#isinstance.
- Continuing a stopped run: the model and optimizer `state_dict` and the epoch in one checkpoint, for resuming training, from the PyTorch tutorial "Saving and Loading Models", https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html; `torch.load` (`map_location`), https://docs.pytorch.org/docs/stable/generated/torch.load.html; `torch.get_rng_state`, https://docs.pytorch.org/docs/stable/generated/torch.get_rng_state.html, and `torch.set_rng_state`, https://docs.pytorch.org/docs/stable/generated/torch.set_rng_state.html; `torch.cuda.get_rng_state_all`, https://docs.pytorch.org/docs/stable/generated/torch.cuda.get_rng_state_all.html, and `torch.cuda.set_rng_state_all`, https://docs.pytorch.org/docs/stable/generated/torch.cuda.set_rng_state_all.html; repeated runs on a GPU, PyTorch documentation "Reproducibility", https://docs.pytorch.org/docs/stable/notes/randomness.html; `os.path.exists`, https://docs.python.org/3/library/os.path.html#os.path.exists, `os.replace` and `os.remove`, https://docs.python.org/3/library/os.html; `wandb.init(mode="disabled")`, W&B documentation, https://docs.coreweave.com/models/ref/python/functions/init, and `wandb.teardown`, https://docs.coreweave.com/models/ref/python/functions/teardown.
- GPU: `torch.device`, `torch.cuda.is_available` and `Tensor.to`, PyTorch documentation "CUDA semantics", https://docs.pytorch.org/docs/stable/notes/cuda.html. HSLU GPUHub and Google Colab from the course slides "Course Project HS26", slide 11. Environment variables on Colab: Google Colab Secrets and `drive.mount`, https://colab.research.google.com/. Repeated runs on a GPU: `torch.backends.cudnn.deterministic`, https://docs.pytorch.org/docs/stable/backends.html, and PyTorch documentation "Reproducibility", https://docs.pytorch.org/docs/stable/notes/randomness.html; `torch.use_deterministic_algorithms` (`warn_only`), https://docs.pytorch.org/docs/stable/generated/torch.use_deterministic_algorithms.html; `CUBLAS_WORKSPACE_CONFIG`, from the warning about RNN functions in `torch.nn.LSTM`, https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html. Setup on Colab: `sys.modules`, https://docs.python.org/3/library/sys.html#sys.modules, and `os.chdir`, https://docs.python.org/3/library/os.html#os.chdir, Python documentation; `%pip`, IPython documentation, https://ipython.readthedocs.io/en/stable/interactive/magics.html.
- Tools: Claude (Anthropic), ChatGPT (OpenAI) and Gemini (Google), see "Use of AI" in README.md.
- Output of W&B: `WANDB_SILENT` (no W&B log lines in the notebook output), W&B documentation "Environment variables", https://docs.coreweave.com/models/track/environment-variables; `os.environ`, Python documentation, https://docs.python.org/3/library/os.html#os.environ.